# Part A: Multimodal document retrieval, Group 22

Executable record of the real lab run (BITS lab, NVIDIA A100, qwen3:8b through Ollama). Run all cells in the lab. Cells call the repository scripts; nothing here is mocked. The ingest takes minutes, so this notebook reads the saved index and the saved results, then runs the integrated demonstration (E10) live.

In [1]:
import csv, json, os, subprocess, sys
from pathlib import Path

ROOT = Path.home() / 'multimodal-rag-group22'
PY = str(Path.home() / 'venv_rag' / 'bin' / 'python')
if not os.path.exists(PY):
    PY = sys.executable

def run(cmd, **kw):
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, **kw)
    print(p.stdout[-3000:] + p.stderr[-800:])
    return p.returncode

print('repo:', ROOT, '| python:', PY)

repo: /home/jovyan/multimodal-rag-group22 | python: /home/jovyan/venv_rag/bin/python


## 1. Environment (GPU, model, packages)

In [2]:
run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'])
run([str(Path.home() / 'ollama012' / 'bin' / 'ollama'), 'list'])
run(['git', 'log', '--oneline', '-1'])

name, driver_version, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 535.309.01, 81920 MiB

NAME        ID              SIZE      MODIFIED       
qwen3:8b    500a1f067a9f    5.2 GB    57 minutes ago    

c07988b Merge pull request #27 from 2025paml539-lab/kishore/e10-docs



0

## 2. Corpus and ingestion (E1)

In [3]:
print(json.dumps(json.load(open(ROOT / 'results' / 'ingest_summary.json')), indent=1)[:1500])
print(json.dumps(json.load(open(ROOT / 'results' / 'e1_inventory.json')), indent=1)[:1200])

{
 "files": 167,
 "failed": [
  [
   "D147",
   "<pymupdf.mupdf.FzErrorArgument; proxy of <Swig Object of type 'mupdf::FzErrorArgument *' at 0x7f4c919410b0> >"
  ]
 ],
 "elements": 11971,
 "chunks": 20339,
 "config": "A",
 "embedder": "st",
 "ocr": false,
 "workers": 5,
 "seconds": 80.6,
 "by_modality": {
  "TEXT": 15093,
  "TABLE": 4869,
  "IMAGE": 377
 }
}
{
 "documents": 166,
 "pages": 8307,
 "elements": {
  "text": 8306,
  "table": 3288,
  "image": 377
 },
 "chunks_config_A": {
  "text": 15093,
  "table": 4869,
  "image": 377
 },
 "by_doc_type": {
  "master_direction": {
   "documents": 129,
   "pages": 7909,
   "text": 7909,
   "table": 3087,
   "image": 261
  },
  "annual_report": {
   "documents": 12,
   "pages": 197,
   "text": 197,
   "table": 128,
   "image": 4
  },
  "slide_deck": {
   "documents": 22,
   "pages": 100,
   "text": 100,
   "table": 23,
   "image": 22
  },
  "booklet_guide": {
   "documents": 3,
   "pages": 101,
   "text": 100,
   "table": 50,
   "image": 90
  

## 3. Integrated demonstration (E10): rate limit, input guardrail, memory, retrieval, prompt, LLM, output guardrail

In [4]:
rc = run([PY, 'scripts/e10_demo.py', '--embedder', 'st', '--llm', 'ollama', '--model', 'qwen3:8b', '--results', 'results/e10'])
print((ROOT / 'results' / 'e10' / 'e10_demo.md').read_text()[:6000])

wrote results/e10/e10_demo.md; 7/8 as expected

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 5895.62it/s]

# E10 integrated demonstration

Index: 20339 chunks. LLM: ollama qwen3:8b. Prompt style: create. Embedder: st.

**7 of 8 steps behaved as expected.** The answers to the four real questions are not compared with gold here; they are shown so the reader can check them against the cited page.

## text question
**Question:** Within what period must the customer submit an OVD with the current address?
**Expected:** output/ok  (golden K-01, D054 p8)
**Stage / verdict:** output / ok
**Top hits:** D102, p.8, D127, p.8, D054, p.8
**Answer:** Within a period of three months [D054, p.8].
**Time:** 1.10 s (guardrails 3.7 ms)
**As expected:** True

## table question
**Question:** In the free-folios table, how many free folios apply to an average credit balance above Rs 50

## 4. Saved results of the final run (30 golden questions)

In [5]:
fin = ROOT / 'results' / 'final_30'
for name in ['e3_retrieval.csv', 'e7_ablation.csv', 'e2_chunking.csv']:
    print('==', name)
    print((fin / name).read_text())
print('== e5_summary (all)'); print(json.dumps(json.load(open(fin / 'e5_summary.json'))['all']))
print('== e6_benchmark'); print(json.dumps({k: v for k, v in json.load(open(fin / 'e6_benchmark.json')).items() if k != 'unstable_queries'}, indent=1))
print('== e5_auto_eval_summary'); print((fin / 'e5_auto_eval_summary.json').read_text())

== e3_retrieval.csv
setup,group,n,recall,mrr
bm25,all,30,0.9333,0.7056
bm25,modality:image,2,1.0,0.75
bm25,modality:table,9,0.8889,0.75
bm25,modality:text,19,0.9474,0.6798
bm25,split:dev,15,1.0,0.7222
bm25,split:test,15,0.8667,0.6889
dense,all,30,0.7667,0.5733
dense,modality:image,2,1.0,1.0
dense,modality:table,9,0.7778,0.6148
dense,modality:text,19,0.7368,0.5088
dense,split:dev,15,0.8,0.6389
dense,split:test,15,0.7333,0.5078
hybrid,all,30,0.8667,0.6661
hybrid,modality:image,2,1.0,1.0
hybrid,modality:table,9,0.8889,0.7167
hybrid,modality:text,19,0.8421,0.607
hybrid,split:dev,15,0.9333,0.63
hybrid,split:test,15,0.8,0.7022
hybrid_text_only,all,30,0.9,0.6606
hybrid_text_only,modality:image,2,1.0,1.0
hybrid_text_only,modality:table,9,0.8889,0.6704
hybrid_text_only,modality:text,19,0.8947,0.6202
hybrid_text_only,split:dev,15,0.9333,0.6356
hybrid_text_only,split:test,15,0.8667,0.6856

== e7_ablation.csv
style,n,correct,faithful,cited_ok,refused,input_tokens_mean,output_tokens_mean,latency_p5

## 5. Memory demonstration (E4)

In [6]:
print((ROOT / 'results' / 'final_30' / 'e4_transcript.md').read_text())

# E4 memory follow-up transcript

LLM: ollama qwen3:8b

## Conversation 1

**User (turn 1):** In the free-folios table, how many free folios apply to an average credit balance above Rs 50,000 and up to Rs 1 lakh?
**Top hits:** D126, p.33 | D126, p.33 | D053, p.29
**Answer:** The number of free folios applicable to an average credit balance above ₹50,000 and up to ₹1 lakh is 5 [D126, p.33].  (expected: 5; correct: True)

**User (turn 2, follow-up):** And for a balance above Rs 1 lakh and up to Rs 2 lakh?
**Rewritten with memory:** And for a balance above ₹1 lakh and up to ₹2 lakh?
**Top hits:** D126, p.33 | D138, p.2 | D039, p.98
**Answer with memory:** For a balance above ₹1 lakh and up to ₹2 lakh, the number of free folios is 10 [D126, p.33].  (expected: 10; correct: True)
**Answer without memory (same follow-up, no history):** The amount of free folios for a balance above Rs 1 lakh and up to Rs 2 lakh is 10. [D003, p.26]  (correct: True)

## Conversation 2

**User (turn 1):** What is

## 6. Offline tests (fake LLM and hash embedder; they check the code, they are not results)

In [7]:
run([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-t', '.'], env={**os.environ, 'PYTHONPATH': str(ROOT / 'src')})

csv")))
/home/jovyan/multimodal-rag-group22/tests/test_b6_runner.py:48: ResourceWarning: unclosed file <_io.TextIOWrapper name='/tmp/tmpj4_e67xk/res/e7_ablation.csv' mode='r' encoding='UTF-8'>
  self.assertEqual(len(list(csv.DictReader(open(res / "e7_ablation.csv")))), 5)
.............................................................../home/jovyan/venv_rag/lib/python3.11/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
s........
----------------------------------------------------------------------
Ran 94 tests in 3.380s

OK (skipped=1)



0